# 08 · Explainable QGAN / Quantum generator သင်ယူပုံ

Install the current source checkout with `python -m pip install -e ".[tutorial]"`.
The published 1.2.0 wheel does not include these APIs. Restart your notebook
kernel after installing. The first cell shows the Python and package paths
and checks that the new APIs are available.

QGAN works on native Windows without PySCF. Do not add the chemistry extra
for this lesson. PySCF requires Linux, macOS or WSL on Windows.
Everything runs locally without a QPU.

The quantum generator produces a Born probability distribution. A classical
discriminator tries to distinguish its outcomes from the target. We alternate
five discriminator updates with one generator update. This is adversarial
training, not direct minimization of distribution error.

Quantum circuit မှ probability များ ထုတ်ပေးပြီး classical discriminator နှင့်
အပြိုင် သင်ယူပါသည်။ အောက်တွင် classical model နှင့်လည်း နှိုင်းယှဉ်ထားပါသည်။


In [ ]:
import sys
import nexqus
print('Kernel Python:', sys.executable)
print('Imported NexQus:', nexqus.__file__)
required = ('QuantumDistribution', 'QGAN', 'distribution_metrics', 'PauliHamiltonian')
missing = [name for name in required if not hasattr(nexqus, name)]
if missing:
    raise ImportError(
        'This kernel imported an older NexQus package without: ' + ', '.join(missing) + '. '
        'Install the current source into this notebook kernel with %pip install --upgrade '
        '"nexqus-sdk[tutorial] @ https://github.com/Inn0m1n4te030/NexQus-SDK/archive/refs/heads/main.zip" '
        'or %pip install -e "/path/to/NexQus-SDK[tutorial]", then restart the kernel and Run All. '
        'The published 1.2.0 wheel lacks these additions; install version 1.4.0rc1 and restart. '
        'QGAN and this fixed-H2 lesson do not require the chemistry extra or PySCF.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.4.0rc1', 'Install NexQus SDK 1.4.0rc1 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


In [ ]:
from nexqus import QuantumDistribution, QGAN, Adam, distribution_metrics
theta = Parameter('theta')
generator = QuantumDistribution(Circuit(1).ry(theta, 0))
model = QGAN(generator, {'theta': 1.3})
target = np.array([.25, .75])
initial = model.probabilities()
model.fit(target, epochs=160)
final = model.probabilities()
print('Initial:', initial, 'Final:', final)
print(distribution_metrics(target, final))
assert distribution_metrics(target, final)['total_variation'] < .03


## Compare against classical, then inspect the generator

A two-category classical model can represent this distribution directly with
probabilities `[1-p, p]`. The target itself is therefore an exact classical
representability reference, not a trained performance comparison. The full
`examples/qgan_distribution.py` example trains both quantum and classical GANs
with identical initial distributions, discriminator initialization, learning
rates and update counts across three seeds; it reports every result.

`jacobian['theta'][i]` tells us the local change in probability of outcome i
per radian, with the current circuit and all other weights fixed. It is not
a global causal explanation or a feature attribution method.


In [ ]:
p, jacobian = generator.value_and_grad(model.parameters)
fig, axes = plt.subplots(1, 2, figsize=(9, 3), layout='constrained')
x = np.arange(2)
axes[0].bar(x-.15, target, width=.3, label='Target / exact classical representation')
axes[0].bar(x+.15, p, width=.3, label='Quantum GAN')
axes[0].set_xticks(x, ['0', '1']); axes[0].set_ylabel('Probability'); axes[0].legend(fontsize=7)
axes[1].plot([r['total_variation'] for r in model.history])
axes[1].set(xlabel='Adversarial update', ylabel='Total variation error')
plt.show()
print('Probability derivatives:', jacobian)
assert abs(jacobian['theta'].sum()) < 1e-10


In [ ]:
from IPython.display import display
display(generator.template.explain(bindings=model.parameters,
    parameter_roles={'theta': 'weight'}, title='QGAN · inspect the trained generator'))
